# `CPPStructurePlot.interactive`

A live, selection-linked explorer: a **site slider** drives a user `predictor` that returns a `df_feat` for that site, and both the **3D structure** and the **`CPPPlot.feature_map`** repaint in place from that one selection. This is the notebook-native version of the deployed cleavage app's per-site explore loop, driven by the real Python model on a live kernel. Rapid slider moves are **debounced**.

This is a `pro` feature (needs `biopython` + `py3Dmol` + `ipywidgets`). Run it in a live Jupyter kernel to drag the slider.

In [1]:
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut

aa.options["verbose"] = False

## The predictor contract

`interactive` is model-agnostic: pass a **`predictor(sequence, p1) -> df_feat`** callable. In a real analysis it wraps `CPP.run` + `ShapModel`/`TreeModel` for the window around `p1`:

```python
def predictor(sequence, p1):
    df_seq = make_window_df_seq(sequence, p1)     # your windowing around the site
    df_feat = cpp.run(labels=labels, ...)         # CPP feature discovery
    sm = aa.ShapModel().fit(X, labels=labels)     # per-sample SHAP impact
    return sm.add_feat_impact(df_feat=df_feat)    # df_feat with a feat_impact column
```

For a self-contained, fast example we use a stub predictor returning a fixed `df_feat`; the linked repaint and controls behave identically.

In [2]:
df_cat = aa.load_scales(name='scales_cat').head(5).reset_index(drop=True)
splits = ['Segment(1,2)', 'Segment(2,2)', 'Segment(1,1)', 'Pattern(C,1)', 'Segment(1,4)']
parts = ['TMD', 'TMD', 'JMD_N', 'TMD', 'JMD_C']
df_feat = pd.DataFrame({
    ut.COL_FEATURE: [f"{parts[i]}-{splits[i]}-{r[ut.COL_SCALE_ID]}" for i, r in df_cat.iterrows()],
    'category': df_cat[ut.COL_CAT], 'subcategory': df_cat[ut.COL_SUBCAT],
    'scale_name': df_cat[ut.COL_SCALE_NAME],
    'abs_auc': [0.2, 0.15, 0.3, 0.1, 0.25], 'abs_mean_dif': [0.3, 0.2, 0.5, 0.4, 0.35],
    'mean_dif': [0.3, -0.2, 0.5, -0.4, 0.25], 'std_test': 0.1, 'std_ref': 0.1,
    'feat_impact': [0.8, -0.5, 1.2, -0.3, 0.6]})

def predictor(sequence, p1):
    # A real predictor re-runs CPP/SHAP for the window at p1; here we return a fixed df_feat.
    return df_feat

# Human lysozyme C (P61626): the AlphaFold structure and its sequence (the slider ranges over it).
sequence = 'MKALIVLGLVLLSVTVQGKVFERCELARTLKRLGMDGYRGISLANWMCLAKWESGYNTRATNYNAGDRSTDYGIFQINSRYWCNDGKTPGAVNACHLSCSALLQDNIADAVACAKRVVRDPQGIRAWVAWRNRCQNRDVRQYVQGCGV'

## Launch the explorer

Drag the **site (P1)** slider to re-predict and repaint both views; the **colour** (`impact`/`plddt`) and **focus** (`whole`/`fade`/`zoom`) dropdowns restyle the structure. `site_to_start` maps the selected site to the structure anchor (default `p1 - jmd_n_len`); `feature_map=False` shows the 3D panel only; `debounce_ms` coalesces rapid moves.

A **highlight** slider links the feature map to the structure: pick a residue in the current window and it lights up in the 3D cartoon (a bold cyan marker) while a vertical line marks its feature-map column — without re-running the predictor. With the `ipympl` backend (`%matplotlib widget`) the feature map is also **clickable** (click a column to drive the same highlight); `ipympl` is optional — the slider is the always-present link.

In [3]:
cpps_plot = aa.CPPStructurePlot(jmd_n_len=10, jmd_c_len=10, verbose=False)
panel = cpps_plot.interactive(predictor=predictor, sequence=sequence, uniprot='P61626',
                        col_imp='feat_impact', tmd_len=10, mode='impact', focus='fade',
                        feature_map=True, init_site=40, debounce_ms=250)
panel

## Custom window geometry and styling

`site_to_start` maps the selected P1 to the structure anchor (the first JMD-N residue; default `p1 - jmd_n_len`). Override it when P1 is not the first TMD residue (e.g. a cleavage P1). `size_by_impact` scales sticks by `|impact|`; `normalize_by_span` switches the per-residue aggregation.

In [4]:
cpps_plot.interactive(predictor=predictor, sequence=sequence, uniprot='P61626', col_imp='feat_impact',
                tmd_len=10, init_site=40, size_by_impact=True, normalize_by_span=False,
                site_to_start=lambda p1: p1 - 8)   # custom anchor mapping

For a one-shot static side-by-side instead of the live explorer, use `CPPStructurePlot.plot_combined`; for the 3D structure alone, `CPPStructurePlot.map_structure`.

In [5]:
# Further parameters. Pick the value column (``col_val``), select a ``chain``,
# frame the camera (``focus_region``), and toggle the SHAP overlay (``shap_plot``).
cpps_plot.interactive(predictor=predictor, sequence=sequence, uniprot="P61626", col_imp="feat_impact",
                      tmd_len=10, init_site=40, col_val="mean_dif", chain="A", focus_region=(40, 49),
                      shap_plot=True)
# ``pdb=<local .pdb/.cif file>`` is the offline alternative to ``uniprot=``.